# Synthetic HR Data Generation

## Purpose
Generate a realistic 2,500-row employee dataset
simulating data from 6 HR systems:
- Workday (HRIS)
- Lattice (Performance Management)
- Culture Amp (Engagement)
- Betterworks (OKR Tracking)
- Culture Amp 360 (360 Feedback)
- Cornerstone (L&D)

## Output
data/raw/employee_performance_raw.csv
- 2,500 employees
- 32 fields
- Realistic correlations and missing values

In [1]:
# ── Import Libraries ──
# pandas → for creating and managing data tables
# numpy  → for math, random numbers, and distributions
# faker  → for generating realistic fake names and IDs
# os     → for handling file paths and folders
import pandas as pd     
import numpy as np      
from faker import Faker 
import os               
import warnings

warnings.filterwarnings('ignore')

# Set random seeds so results are reproducible
# This means every time you run this code
# you get the exact same dataset
np.random.seed(42)
fake = Faker()
Faker.seed(42)

print("Libraries imported successfully") 

Libraries imported successfully


In [2]:
# ── Configuration ──
N_EMPLOYEES = 2500  # Total number of employees to generate

# Department names (like a real mid-size company)
DEPARTMENTS = [
    'Engineering', 'Sales', 'HR', 'Finance',
    'Marketing', 'Operations', 'Product', 'Customer Success'
]

# Job roles for each department
JOB_ROLES = {
    'Engineering':       ['Software Engineer', 'Data Engineer',
                          'DevOps Engineer', 'ML Engineer'],
    'Sales':             ['Account Executive', 'Sales Development Rep',
                          'Sales Manager'],
    'HR':                ['HR Business Partner', 'Recruiter',
                          'L&D Specialist', 'Compensation Analyst'],
    'Finance':           ['Financial Analyst', 'Accountant',
                          'Finance Manager', 'FP&A Analyst'],
    'Marketing':         ['Marketing Analyst', 'Content Strategist',
                          'Growth Manager'],
    'Operations':        ['Operations Analyst', 'Supply Chain Specialist',
                          'Ops Manager'],
    'Product':           ['Product Manager', 'UX Researcher',
                          'Product Analyst'],
    'Customer Success':  ['CS Manager', 'CS Specialist',
                          'Onboarding Manager']
}

# Job levels from junior to senior
JOB_LEVELS    = ['IC1', 'IC2', 'IC3', 'M1', 'M2', 'Director']

# How many employees at each level
# Most companies are pyramid shaped:
# many junior, few senior
LEVEL_WEIGHTS = [0.25, 0.30, 0.20, 0.12, 0.08, 0.05]

# Numeric version of each level (used in calculations)
LEVEL_MAP = {
    'IC1': 1, 'IC2': 2, 'IC3': 3,
    'M1': 4,  'M2': 5,  'Director': 6
}

print(f"Configuration set")
print(f"Employees to generate : {N_EMPLOYEES}")
print(f"Departments           : {len(DEPARTMENTS)}")
print(f"Job levels            : {JOB_LEVELS}") 

Configuration set
Employees to generate : 2500
Departments           : 8
Job levels            : ['IC1', 'IC2', 'IC3', 'M1', 'M2', 'Director']


In [3]:
# ── Identifiers and Demographics ──
# Simulates: Workday / SAP HRIS data

# ── a: Assign department, role, level ──
department = np.random.choice(DEPARTMENTS, N_EMPLOYEES)

role = [
    np.random.choice(JOB_ROLES[dept])
    for dept in department
]

level = np.random.choice(
    JOB_LEVELS,
    N_EMPLOYEES,
    p=LEVEL_WEIGHTS
)

# Convert level to number for calculations
level_num = np.array([LEVEL_MAP[l] for l in level])

# ── b: Create hidden talent score ── 
# This is the secret engine behind all correlations
# High latent_performance → high OKR, high rating, high engagement
# Low latent_performance  → opposite
# Real world: some people are genuinely higher performers
latent_performance = np.random.normal(0, 1, N_EMPLOYEES)

# ── c: Demographics ── 
age = np.clip(
    np.random.normal(35, 8, N_EMPLOYEES),
    22, 62
).astype(int)

gender = np.random.choice(
    ['Male', 'Female', 'Non-Binary'],
    N_EMPLOYEES,
    p=[0.52, 0.44, 0.04]
)

education = np.random.choice(
    ["Bachelor's", "Master's", "PhD", "Associate's", "High School"],
    N_EMPLOYEES,
    p=[0.45, 0.35, 0.08, 0.08, 0.04]
)

location_type = np.random.choice(
    ['Remote', 'Hybrid', 'On-Site'],
    N_EMPLOYEES,
    p=[0.30, 0.45, 0.25]
)

# ── d: Employee and Manager IDs ── 
employee_ids = [f"EMP{str(i).zfill(5)}" for i in range(1, N_EMPLOYEES + 1)]
manager_pool = [f"MGR{str(i).zfill(4)}" for i in range(1, 201)]
manager_ids  = np.random.choice(manager_pool, N_EMPLOYEES)

# ── e: Tenure ── 
# Exponential distribution = most employees are newer
# Few have very long tenure (realistic for most companies)
years_at_company = np.clip(
    np.random.exponential(4, N_EMPLOYEES),
    0.5, 25
).round(1)

years_since_promo = np.clip(
    years_at_company * np.random.uniform(0.3, 0.9, N_EMPLOYEES),
    0, 10
).round(1)

print(f"Demographics generated")
print(f"Sample Employee IDs : {employee_ids[:3]}")
print(f"Department spread   :")
dept_series = pd.Series(department)
print(dept_series.value_counts().to_string()) 

Demographics generated
Sample Employee IDs : ['EMP00001', 'EMP00002', 'EMP00003']
Department spread   :
Engineering         329
Marketing           327
HR                  314
Sales               309
Finance             307
Product             306
Customer Success    305
Operations          303


In [4]:
# ── Performance Ratings ──
# Simulates: Lattice / SuccessFactors Performance Management

# ── Current Performance Rating (1 to 5 scale) ── 
# Formula: base score 3.0
#        + talent signal (stronger talent = higher rating)
#        + random noise (managers are not perfectly objective)
raw_rating = (
    3.0
    + 0.8 * latent_performance
    + np.random.normal(0, 0.4, N_EMPLOYEES)
)

# Round to nearest 0.5 (companies use half-point scales)
# Clip to valid range 1 to 5
performance_rating = np.clip(
    np.round(raw_rating * 2) / 2,
    1, 5
)

# ── Historical Rating (prior 2 cycles average) ── 
# Similar to current but with some drift
# High performers tend to have consistently high history
historical_rating_avg = np.clip(
    performance_rating
    - np.random.uniform(-0.5, 0.5, N_EMPLOYEES)
    + np.random.normal(0, 0.2, N_EMPLOYEES),
    1, 5
).round(2)

# ── Calibration Adjusted Flag ── 
# 18% of ratings get adjusted during calibration meetings
# This is realistic — managers recalibrate each other
calibration_adjusted = np.random.choice(
    [0, 1],
    N_EMPLOYEES,
    p=[0.82, 0.18]
)

print(f"Performance ratings generated")
print(f"\nRating Distribution:")
rating_series = pd.Series(performance_rating)
print(rating_series.value_counts().sort_index().to_string())
print(f"\nAverage Rating : {performance_rating.mean():.2f}")
print(f"% High Performers (4.0+) : {(performance_rating >= 4.0).mean()*100:.1f}%")
print(f"% Low Performers  (2.0-) : {(performance_rating <= 2.0).mean()*100:.1f}%")
print(f"Calibration Adjusted     : {calibration_adjusted.mean()*100:.1f}%") 

Performance ratings generated

Rating Distribution:
1.0     75
1.5    145
2.0    331
2.5    467
3.0    526
3.5    465
4.0    273
4.5    149
5.0     69

Average Rating : 2.97
% High Performers (4.0+) : 19.6%
% Low Performers  (2.0-) : 22.0%
Calibration Adjusted     : 17.1%


In [5]:
# ── 360 Feedback ──
# Simulates: Culture Amp 360 / Reflektive

# ── Self Rating ──
# People rate themselves slightly higher than reality
# Average self-inflation: +0.3 points
self_rating = np.clip(
    performance_rating + np.random.normal(0.3, 0.4, N_EMPLOYEES),
    1, 5
).round(2)

# ── Peer Average Rating ──
# Peers are slightly more objective than self
# Small negative bias: -0.1 (peers tend to be slightly critical)
peer_avg_rating = np.clip(
    performance_rating + np.random.normal(-0.1, 0.35, N_EMPLOYEES),
    1, 5
).round(2)

# ── Subordinate Average Rating ──
# Only managers have direct reports who can rate them
# IC1 and IC2 = individual contributors = no subordinates
# So their subordinate rating is NaN (blank/missing)

has_direct_reports = np.isin(level, ['M1', 'M2', 'Director'])

sub_rating_raw = np.clip(
    performance_rating + np.random.normal(0.05, 0.45, N_EMPLOYEES),
    1, 5
).round(2)

# Apply NaN for non-managers
subordinate_avg_rating = np.where(
    has_direct_reports,
    sub_rating_raw,
    np.nan
)

# ── Overall 360 Score ──
# Average of all three ratings
# Uses nanmean to handle the missing subordinate values
overall_360 = np.nanmean(
    np.stack([self_rating, peer_avg_rating, sub_rating_raw], axis=1),
    axis=1
).round(2)

# ── Self Other Gap ──
# Positive = overestimates self (common)
# Negative = underestimates self (impostor syndrome)
self_other_gap = (self_rating - peer_avg_rating).round(2)

# ── Leadership and Collaboration 360 ──
# Leadership score: higher for managers (they are rated on it more)
leadership_360 = np.clip(
    performance_rating * 0.7
    + level_num * 0.1
    + np.random.normal(0, 0.5, N_EMPLOYEES),
    1, 5
).round(2)

# Collaboration score: tied to peer ratings
collaboration_360 = np.clip(
    peer_avg_rating * 0.8
    + np.random.normal(0, 0.3, N_EMPLOYEES),
    1, 5
).round(2)

# ── Add realistic missing values to 360 fields ──
# In real life: not everyone completes 360 surveys
# About 6% of peer and leadership ratings are missing
def add_missing(array, pct=0.06):
    arr = array.copy().astype(float)
    missing_idx = np.random.choice(
        len(arr),
        size=int(pct * len(arr)),
        replace=False
    )
    arr[missing_idx] = np.nan
    return arr

peer_avg_rating    = add_missing(peer_avg_rating)
overall_360        = add_missing(overall_360)
leadership_360     = add_missing(leadership_360)
collaboration_360  = add_missing(collaboration_360)

print(f"360 Feedback generated")
print(f"Average Self Rating      : {np.nanmean(self_rating):.2f}")
print(f"Average Peer Rating      : {np.nanmean(peer_avg_rating):.2f}")
print(f"Average Overall 360      : {np.nanmean(overall_360):.2f}")
print(f"Self-Other Gap (avg)     : {np.nanmean(self_other_gap):.2f}")
print(f"Subordinate Rating (NaN) : {np.isnan(subordinate_avg_rating).sum()} employees (ICs)")
print(f"Missing Peer Ratings     : {np.isnan(peer_avg_rating).sum()}") 

360 Feedback generated
Average Self Rating      : 3.26
Average Peer Rating      : 2.87
Average Overall 360      : 3.06
Self-Other Gap (avg)     : 0.39
Subordinate Rating (NaN) : 1888 employees (ICs)
Missing Peer Ratings     : 150


In [6]:
# ── OKR / Goals ──
# Simulates: Betterworks / Ally.io / Lattice Goals

# ── Number of OKRs Assigned ──
# Senior employees get more OKRs (more responsibilities)
# level_num ranges from 1 (IC1) to 6 (Director)
num_okrs_assigned = np.clip(
    (level_num * 1.2 + np.random.normal(3, 1, N_EMPLOYEES)).astype(int),
    1, 8
)

# ── OKR Completion Percentage ──
# Strongly tied to latent performance (correlation ~0.60)
# Base completion: 72%
# High performers complete more OKRs
okr_completion_pct = np.clip(
    0.72
    + 0.15 * latent_performance
    + np.random.normal(0, 0.08, N_EMPLOYEES),
    0.1, 1.0
)

# ── Weighted Goal Attainment ──
# Some OKRs are worth more than others (weighted)
# Weighted attainment is slightly different from raw completion
weighted_goal_attainment = np.clip( 
    okr_completion_pct * np.random.uniform(0.85, 1.10, N_EMPLOYEES),
    0.05, 1.0
)

# Convert to percentage (multiply by 100)
okr_completion_pct       = (okr_completion_pct * 100).round(1)
weighted_goal_attainment = (weighted_goal_attainment * 100).round(1)

print(f"OKR data generated")
print(f"Average OKR Completion     : {okr_completion_pct.mean():.1f}%")
print(f"Average Weighted Attainment: {weighted_goal_attainment.mean():.1f}%")
print(f"Average OKRs Assigned      : {num_okrs_assigned.mean():.1f}")
print(f"\nOKR Completion Distribution:")
okr_bands = pd.cut(
    okr_completion_pct,
    bins=[0, 50, 70, 85, 100],
    labels=['Below 50%', '50-70%', '70-85%', '85%+']
)
print(pd.Series(okr_bands).value_counts().sort_index().to_string()) 

OKR data generated
Average OKR Completion     : 71.0%
Average Weighted Attainment: 69.0%
Average OKRs Assigned      : 5.5

OKR Completion Distribution:
Below 50%    264
50-70%       906
70-85%       822
85%+         508


In [7]:
# ── Engagement and Wellbeing ──
# Simulates: Culture Amp / Glint Engagement Surveys

# ── Engagement Score (0 to 100) ──
# Correlated with latent performance
# High performers tend to be more engaged (not always but often)
engagement_score = np.clip(
    55
    + 12 * latent_performance
    + np.random.normal(0, 10, N_EMPLOYEES),
    10, 100
).round(1)

# ── Job Satisfaction (1 to 5) ──
job_satisfaction = np.clip(
    3.0
    + 0.5 * latent_performance
    + np.random.normal(0, 0.7, N_EMPLOYEES),
    1, 5
).round(2)

# ── Work Life Balance Rating (1 to 5) ──
# Less correlated with performance — more random
work_life_balance = np.clip(
    3.2 + np.random.normal(0, 0.8, N_EMPLOYEES),
    1, 5
).round(2)

# ── Overtime Hours Monthly ──
# Senior employees work more overtime
# Exponential distribution = most people work some overtime
# Few work extreme overtime
overtime_monthly = np.clip(
    10
    + np.random.exponential(8, N_EMPLOYEES)
    + level_num * 2,
    0, 80
).round(1)

# ── Burnout Risk ──
# Driven by: overtime (40%) + low engagement (40%) + random (20%)
# Higher overtime + lower engagement = higher burnout risk
burnout_score = (
    0.4 * (overtime_monthly / 80)
    + 0.4 * (1 - engagement_score / 100)
    + 0.2 * np.random.uniform(0, 1, N_EMPLOYEES)
)

# Categorize into Low / Medium / High
burnout_risk = pd.cut(
    burnout_score,
    bins=[0, 0.33, 0.60, 1.01],
    labels=['Low', 'Medium', 'High']
).astype(str)

print(f"Engagement data generated")
print(f"Average Engagement Score : {engagement_score.mean():.1f}/100")
print(f"Average Job Satisfaction : {job_satisfaction.mean():.2f}/5")
print(f"Average Work-Life Balance: {work_life_balance.mean():.2f}/5")
print(f"Average Overtime/Month   : {overtime_monthly.mean():.1f} hours")
print(f"\nBurnout Risk Distribution:")
print(pd.Series(burnout_risk).value_counts().to_string()) 

Engagement data generated
Average Engagement Score : 54.2/100
Average Job Satisfaction : 2.98/5
Average Work-Life Balance: 3.17/5
Average Overtime/Month   : 23.3 hours

Burnout Risk Distribution:
Medium    1833
Low        614
High        53


In [8]:
# ── Productivity and Compensation ──
# Simulates: Cornerstone LMS + HRIS Compensation Data

# ── Training Hours Last Year ──
# High performers invest more in learning
training_hours = np.clip(
    20
    + 15 * latent_performance
    + np.random.normal(0, 10, N_EMPLOYEES),
    0, 120
).round(0).astype(int)

# ── Average Monthly Hours Worked ──
# Base: 160 hours (standard work month)
# Plus overtime hours
avg_monthly_hours = np.clip(
    160
    + overtime_monthly
    + np.random.normal(0, 10, N_EMPLOYEES),
    100, 280
).round(0).astype(int)

# ── Absenteeism Days Per Year ──
# Negative binomial = realistic sick day distribution
# Most employees miss a few days, some miss many
absenteeism_days = np.clip(
    np.random.negative_binomial(2, 0.5, N_EMPLOYEES),
    0, 30
).astype(int)

# ── Projects Handled ──
# Senior employees handle more projects
projects_handled = np.clip(
    (level_num * 1.5 + np.random.normal(2, 1.5, N_EMPLOYEES)).astype(int),
    1, 12
)

# ── Compensation ──
# Salary hike: better performers get bigger raises
salary_hike_pct = np.clip(
    5
    + 3 * latent_performance
    + np.random.normal(0, 2, N_EMPLOYEES),
    0, 25
).round(1)

# Bonus payout as proportion of target bonus
# 1.0 = exactly target, 1.5 = 150% of target
bonus_payout_pct = np.clip(
    0.6
    + 0.3 * latent_performance
    + np.random.normal(0, 0.15, N_EMPLOYEES),
    0, 1.5
).round(2)

# ── High Potential Flag ──
# Employee is flagged as high potential IF:
# - Rating is 4.0 or above AND
# - OKR completion is 75% or above AND
# - At least 1.5 years at company AND
# - Some randomness (not all HiPos are identified)
high_potential_flag = (
    (performance_rating >= 4.0)
    & (okr_completion_pct >= 75)
    & (years_at_company >= 1.5)
    & (np.random.random(N_EMPLOYEES) > 0.3)
).astype(int)

# ── PIP History Flag ──
# Employee was previously on Performance Improvement Plan IF:
# - Current rating is 2.0 or below AND
# - Historical rating was also low AND
# - Some randomness (not all low performers had PIPs)
pip_history_flag = (
    (performance_rating <= 2.0)
    & (historical_rating_avg <= 2.5)
    & (np.random.random(N_EMPLOYEES) > 0.5)
).astype(int)

print(f"Productivity and compensation data generated")
print(f"Average Training Hours  : {training_hours.mean():.1f} hrs/year")
print(f"Average Monthly Hours   : {avg_monthly_hours.mean():.1f} hrs/month")
print(f"Average Absenteeism     : {absenteeism_days.mean():.1f} days/year")
print(f"Average Projects        : {projects_handled.mean():.1f}")
print(f"Average Salary Hike     : {salary_hike_pct.mean():.1f}%")
print(f"High Potential Employees: {high_potential_flag.sum()} ({high_potential_flag.mean()*100:.1f}%)")
print(f"PIP History Employees   : {pip_history_flag.sum()} ({pip_history_flag.mean()*100:.1f}%)") 

Productivity and compensation data generated
Average Training Hours  : 21.0 hrs/year
Average Monthly Hours   : 183.2 hrs/month
Average Absenteeism     : 2.0 days/year
Average Projects        : 5.4
Average Salary Hike     : 5.0%
High Potential Employees: 226 (9.0%)
PIP History Employees   : 267 (10.7%)


In [9]:
# ── Combine All Fields Into One DataFrame ──
df = pd.DataFrame({

    # ── Identifiers (from Workday) ──
    'EmployeeID'               : employee_ids,
    'Department'               : department,
    'JobRole'                  : role,
    'JobLevel'                 : level,
    'ManagerID'                : manager_ids,
    'LocationType'             : location_type,
    'Age'                      : age,
    'Gender'                   : gender,
    'EducationLevel'           : education,
    'YearsAtCompany'           : years_at_company,
    'YearsSinceLastPromotion'  : years_since_promo,

    # ── Performance (from Lattice) ──
    'PerformanceRating'        : performance_rating,
    'HistoricalRatingAvg'      : historical_rating_avg,
    'CalibrationAdjustedFlag'  : calibration_adjusted,

    # ── 360 Feedback (from Culture Amp 360) ──
    'SelfRating'               : self_rating,
    'PeerAvgRating'            : peer_avg_rating,
    'SubordinateAvgRating'     : subordinate_avg_rating,
    'Overall360Score'          : overall_360,
    'SelfOtherGap'             : self_other_gap,
    'Leadership360'            : leadership_360,
    'Collaboration360'         : collaboration_360,

    # ── OKR Data (from Betterworks) ──
    'OKRCompletionPct'         : okr_completion_pct,
    'NumOKRsAssigned'          : num_okrs_assigned,
    'WeightedGoalAttainment'   : weighted_goal_attainment,

    # ── Engagement (from Culture Amp Surveys) ──
    'EngagementScore'          : engagement_score,
    'JobSatisfaction'          : job_satisfaction,
    'WorkLifeBalanceRating'    : work_life_balance,
    'BurnoutRisk'              : burnout_risk,

    # ── Productivity (from Cornerstone + HRIS) ──
    'TrainingHoursLastYear'    : training_hours,
    'OvertimeHoursMonthly'     : overtime_monthly,
    'AbsenteeismDays'          : absenteeism_days,
    'AvgMonthlyHours'          : avg_monthly_hours,
    'ProjectsHandled'          : projects_handled,

    # ── Compensation (from HRIS) ──
    'PercentSalaryHikeLast'    : salary_hike_pct,
    'BonusPayoutPct'           : bonus_payout_pct,
    'YearsSinceLastPromotion'  : years_since_promo,
    'HighPotentialFlag'        : high_potential_flag,
    'PIPHistoryFlag'           : pip_history_flag,

})

print(f"DataFrame created successfully")
print(f"Rows    : {df.shape[0]} employees")
print(f"Columns : {df.shape[1]} fields")
print(f"\nFirst 3 rows preview:")
print(df.head(3).to_string()) 

DataFrame created successfully
Rows    : 2500 employees
Columns : 37 fields

First 3 rows preview:
  EmployeeID Department            JobRole  JobLevel ManagerID LocationType  Age  Gender EducationLevel  YearsAtCompany  YearsSinceLastPromotion  PerformanceRating  HistoricalRatingAvg  CalibrationAdjustedFlag  SelfRating  PeerAvgRating  SubordinateAvgRating  Overall360Score  SelfOtherGap  Leadership360  Collaboration360  OKRCompletionPct  NumOKRsAssigned  WeightedGoalAttainment  EngagementScore  JobSatisfaction  WorkLifeBalanceRating BurnoutRisk  TrainingHoursLastYear  OvertimeHoursMonthly  AbsenteeismDays  AvgMonthlyHours  ProjectsHandled  PercentSalaryHikeLast  BonusPayoutPct  HighPotentialFlag  PIPHistoryFlag
0   EMP00001    Product      UX Researcher        M1   MGR0171       Remote   31  Female       Master's             1.4                      1.2                1.5                 2.04                        0        1.49           2.10                  2.04             1.88     

In [10]:
# ── Data Quality Check ──

print("-" * 55)
print("DATA QUALITY REPORT")
print("-" * 55)

# Shape
print(f"\nSHAPE")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# Missing values
print(f"\nMISSING VALUES (only columns with missing shown)")
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    for col, count in missing_cols.items():
        pct = count / len(df) * 100
        print(f"{col:<30} {count:>4} missing ({pct:.1f}%)")
else:
    print("No missing values found")

# Data types
print(f"\nCOLUMN DATA TYPES")
for col, dtype in df.dtypes.items():
    print(f"{col:<35} {str(dtype)}")

# Key statistics
print(f"\nKEY STATISTICS")
print(f"Avg Performance Rating : {df['PerformanceRating'].mean():.2f}")
print(f"Avg OKR Completion     : {df['OKRCompletionPct'].mean():.1f}%")
print(f"Avg Engagement Score   : {df['EngagementScore'].mean():.1f}")
print(f"High Potential Count   : {df['HighPotentialFlag'].sum()}")
print(f"PIP History Count      : {df['PIPHistoryFlag'].sum()}")

# Correlation check
print(f"\nKEY CORRELATIONS WITH PERFORMANCE RATING")
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()['PerformanceRating'].drop('PerformanceRating')
top_corr = corr.abs().sort_values(ascending=False).head(8)
for col, val in top_corr.items():
    direction = "+" if corr[col] > 0 else "-"
    print(f"{direction} {col:<35} r = {corr[col]:.3f}")
    
print("\nData quality check complete") 

-------------------------------------------------------
DATA QUALITY REPORT
-------------------------------------------------------

SHAPE
Rows    : 2500
Columns : 37

MISSING VALUES (only columns with missing shown)
PeerAvgRating                   150 missing (6.0%)
SubordinateAvgRating           1888 missing (75.5%)
Overall360Score                 150 missing (6.0%)
Leadership360                   150 missing (6.0%)
Collaboration360                150 missing (6.0%)

COLUMN DATA TYPES
EmployeeID                          str
Department                          str
JobRole                             str
JobLevel                            str
ManagerID                           str
LocationType                        str
Age                                 int64
Gender                              str
EducationLevel                      str
YearsAtCompany                      float64
YearsSinceLastPromotion             float64
PerformanceRating                   float64
HistoricalRati

In [11]:
# ── Save to CSV ──

# Build the file path
# This works regardless of where your project folder is
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
output_path  = os.path.join(project_root, 'data', 'raw',
                             'employee_performance_raw.csv')

# Create the folder if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the CSV
df.to_csv(output_path, index=False)

print(f"Dataset saved successfully!")
print(f"\nFile location:")
print(f"{output_path}")
print(f"\nFile details:")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nColumn names saved:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:>2}. {col}") 

Dataset saved successfully!

File location:
C:\Users\ganti_kvd0xe3\OneDrive\Sathya\employee_performance_analytics\data\raw\employee_performance_raw.csv

File details:
Rows    : 2,500
Columns : 37

Column names saved:
    1. EmployeeID
    2. Department
    3. JobRole
    4. JobLevel
    5. ManagerID
    6. LocationType
    7. Age
    8. Gender
    9. EducationLevel
   10. YearsAtCompany
   11. YearsSinceLastPromotion
   12. PerformanceRating
   13. HistoricalRatingAvg
   14. CalibrationAdjustedFlag
   15. SelfRating
   16. PeerAvgRating
   17. SubordinateAvgRating
   18. Overall360Score
   19. SelfOtherGap
   20. Leadership360
   21. Collaboration360
   22. OKRCompletionPct
   23. NumOKRsAssigned
   24. WeightedGoalAttainment
   25. EngagementScore
   26. JobSatisfaction
   27. WorkLifeBalanceRating
   28. BurnoutRisk
   29. TrainingHoursLastYear
   30. OvertimeHoursMonthly
   31. AbsenteeismDays
   32. AvgMonthlyHours
   33. ProjectsHandled
   34. PercentSalaryHikeLast
   35. BonusPay

In [12]:
# ── Final Verification ──
# Read the saved file back to confirm it saved correctly

df_check = pd.read_csv(output_path)

print("-" * 50)
print("FINAL VERIFICATION - FILE SAVED CORRECTLY")
print("-" * 50)
print(f"Rows loaded back : {df_check.shape[0]:,}")
print(f"Columns loaded   : {df_check.shape[1]}")
print(f"\nSample data (first 5 rows, first 6 columns):")
print(df_check.iloc[:5, :6].to_string(index=False))
print("-" * 50) 
print(f"File: data/raw/employee_performance_raw.csv")
print("-" * 50) 

--------------------------------------------------
FINAL VERIFICATION - FILE SAVED CORRECTLY
--------------------------------------------------
Rows loaded back : 2,500
Columns loaded   : 37

Sample data (first 5 rows, first 6 columns):
EmployeeID Department           JobRole JobLevel ManagerID LocationType
  EMP00001    Product     UX Researcher       M1   MGR0171       Remote
  EMP00002    Finance   Finance Manager Director   MGR0137       Hybrid
  EMP00003  Marketing Marketing Analyst       M1   MGR0019       Hybrid
  EMP00004    Product     UX Researcher       M1   MGR0140      On-Site
  EMP00005         HR         Recruiter      IC1   MGR0155       Remote
--------------------------------------------------
File: data/raw/employee_performance_raw.csv
--------------------------------------------------


In [13]:
# ── FIX: Force create 9-Box edge case employees ──
# These employees represent realistic but rare combinations
# that our correlated data generation missed

import pandas as pd
import numpy as np
import os

notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

# Load existing dataset
raw_path = os.path.join(project_root, 'data', 'raw',
                        'employee_performance_raw.csv')
df_existing = pd.read_csv(raw_path)

np.random.seed(99)

# ── GROUP 1: Solid Contributors ──
# High Performance (4.0+) BUT Low Readiness (<0.35)
# Real world example: Technical expert who does not want
# to be promoted — loves their current IC role
# OR: New high performer not yet eligible for promotion

solid_contributors = pd.DataFrame({
    'EmployeeID'              : [f"EMP0{i}" for i in range(9001, 9051)],
    'Department'              : np.random.choice(
                                    ['Engineering','Finance',
                                     'Product','Operations'], 50),
    'JobRole'                 : ['Software Engineer'] * 50,
    'JobLevel'                : np.random.choice(
                                    ['IC2','IC3'], 50),
    'ManagerID'               : [f"MGR{str(i).zfill(4)}"
                                  for i in np.random.randint(1,201,50)],
    'LocationType'            : np.random.choice(
                                    ['Remote','Hybrid','On-Site'], 50),
    'Age'                     : np.random.randint(25, 40, 50),
    'Gender'                  : np.random.choice(
                                    ['Male','Female','Non-Binary'], 50),
    'EducationLevel'          : np.random.choice(
                                    ["Bachelor's","Master's"], 50),
    'YearsAtCompany'          : np.random.uniform(0.5, 1.5, 50).round(1),
    'YearsSinceLastPromotion' : np.random.uniform(0.3, 1.0, 50).round(1),

    # HIGH performance (4.0 to 5.0)
    'PerformanceRating'       : np.random.uniform(4.0, 5.0, 50).round(1),
    'HistoricalRatingAvg'     : np.random.uniform(3.5, 4.5, 50).round(2),
    'CalibrationAdjustedFlag' : np.random.choice([0,1], 50, p=[0.85,0.15]),

    # Normal 360 scores
    'SelfRating'              : np.random.uniform(3.8, 4.8, 50).round(2),
    'PeerAvgRating'           : np.random.uniform(3.5, 4.5, 50).round(2),
    'SubordinateAvgRating'    : np.nan,
    'Overall360Score'         : np.random.uniform(3.6, 4.6, 50).round(2),
    'SelfOtherGap'            : np.random.uniform(0.1, 0.5, 50).round(2),
    'Leadership360'           : np.random.uniform(3.0, 4.0, 50).round(2),
    'Collaboration360'        : np.random.uniform(3.5, 4.5, 50).round(2),

    # LOW OKR (new role, still ramping up goals)
    'OKRCompletionPct'        : np.random.uniform(30, 55, 50).round(1),
    'NumOKRsAssigned'         : np.random.randint(1, 3, 50),
    'WeightedGoalAttainment'  : np.random.uniform(25, 50, 50).round(1),

    # Good engagement
    'EngagementScore'         : np.random.uniform(55, 75, 50).round(1),
    'JobSatisfaction'         : np.random.uniform(3.2, 4.5, 50).round(2),
    'WorkLifeBalanceRating'   : np.random.uniform(3.0, 4.5, 50).round(2),
    'BurnoutRisk'             : np.random.choice(
                                    ['Low','Medium'], 50, p=[0.7,0.3]),

    # Normal productivity
    'TrainingHoursLastYear'   : np.random.randint(20, 60, 50),
    'OvertimeHoursMonthly'    : np.random.uniform(5, 20, 50).round(1),
    'AbsenteeismDays'         : np.random.randint(0, 5, 50),
    'AvgMonthlyHours'         : np.random.randint(160, 185, 50),
    'ProjectsHandled'         : np.random.randint(2, 5, 50),
    'PercentSalaryHikeLast'   : np.random.uniform(8, 15, 50).round(1),
    'BonusPayoutPct'          : np.random.uniform(0.8, 1.3, 50).round(2),

    # SHORT tenure = low readiness (main reason)
    'HighPotentialFlag'       : np.zeros(50, dtype=int),
    'PIPHistoryFlag'          : np.zeros(50, dtype=int),
})

# ── GROUP 2: Inconsistent HiPos ──
# Low Performance (<3.0) BUT High Readiness (>0.55)
# Real world example: High potential employee in a
# stretch role that is too big for them right now
# OR: Manager transition — demoted temporarily
# OR: Personal circumstances affecting performance

inconsistent_hipo = pd.DataFrame({
    'EmployeeID'              : [f"EMP0{i}" for i in range(9051, 9101)],
    'Department'              : np.random.choice(
                                    ['Sales','HR',
                                     'Marketing','Customer Success'], 50),
    'JobRole'                 : ['Product Manager'] * 50,
    'JobLevel'                : np.random.choice(['M1','IC3'], 50),
    'ManagerID'               : [f"MGR{str(i).zfill(4)}"
                                  for i in np.random.randint(1,201,50)],
    'LocationType'            : np.random.choice(
                                    ['Remote','Hybrid','On-Site'], 50),
    'Age'                     : np.random.randint(28, 45, 50),
    'Gender'                  : np.random.choice(
                                    ['Male','Female','Non-Binary'], 50),
    'EducationLevel'          : np.random.choice(
                                    ["Bachelor's","Master's","PhD"], 50),
    'YearsAtCompany'          : np.random.uniform(4.0, 8.0, 50).round(1),
    'YearsSinceLastPromotion' : np.random.uniform(3.5, 6.0, 50).round(1),

    # LOW performance (1.5 to 2.9)
    'PerformanceRating'       : np.random.uniform(1.5, 2.9, 50).round(1),
    'HistoricalRatingAvg'     : np.random.uniform(3.5, 4.5, 50).round(2),
    'CalibrationAdjustedFlag' : np.random.choice([0,1], 50, p=[0.7,0.3]),

    # HIGH 360 scores (peers see their potential)
    'SelfRating'              : np.random.uniform(3.5, 4.5, 50).round(2),
    'PeerAvgRating'           : np.random.uniform(3.8, 4.8, 50).round(2),
    'SubordinateAvgRating'    : np.random.uniform(3.5, 4.5, 50).round(2),
    'Overall360Score'         : np.random.uniform(3.7, 4.7, 50).round(2),
    'SelfOtherGap'            : np.random.uniform(-0.5, 0.2, 50).round(2),
    'Leadership360'           : np.random.uniform(3.8, 4.8, 50).round(2),
    'Collaboration360'        : np.random.uniform(3.8, 4.8, 50).round(2),

    # HIGH OKR historically
    'OKRCompletionPct'        : np.random.uniform(75, 92, 50).round(1),
    'NumOKRsAssigned'         : np.random.randint(4, 8, 50),
    'WeightedGoalAttainment'  : np.random.uniform(70, 90, 50).round(1),

    # LOW engagement (struggling in current role)
    'EngagementScore'         : np.random.uniform(20, 40, 50).round(1),
    'JobSatisfaction'         : np.random.uniform(1.5, 2.5, 50).round(2),
    'WorkLifeBalanceRating'   : np.random.uniform(1.5, 2.8, 50).round(2),
    'BurnoutRisk'             : np.random.choice(
                                    ['Medium','High'], 50, p=[0.4,0.6]),

    # High overtime (struggling = working more hours)
    'TrainingHoursLastYear'   : np.random.randint(40, 80, 50),
    'OvertimeHoursMonthly'    : np.random.uniform(30, 60, 50).round(1),
    'AbsenteeismDays'         : np.random.randint(5, 15, 50),
    'AvgMonthlyHours'         : np.random.randint(190, 240, 50),
    'ProjectsHandled'         : np.random.randint(4, 8, 50),
    'PercentSalaryHikeLast'   : np.random.uniform(3, 7, 50).round(1),
    'BonusPayoutPct'          : np.random.uniform(0.4, 0.7, 50).round(2),

    # HIGH potential flag (manager recognizes hidden talent)
    'HighPotentialFlag'       : np.ones(50, dtype=int),
    'PIPHistoryFlag'          : np.zeros(50, dtype=int),
})

# ── Combine and save ──
df_combined = pd.concat(
    [df_existing, solid_contributors, inconsistent_hipo],
    ignore_index=True
)

df_combined.to_csv(raw_path, index=False)

print(f"Edge case employees added successfully")
print(f"\nOriginal employees : {len(df_existing):,}")
print(f"Solid Contributors   : {len(solid_contributors)}")
print(f"Inconsistent HiPos   : {len(inconsistent_hipo)}")
print(f"Total employees now  : {len(df_combined):,}")
print(f"\nThese employees represent:")
print(f"Solid Contributors → High performer, short tenure,")
print(f"low OKR, not yet promo eligible")
print(f"Inconsistent HiPos → Low current performance but")
print(f"strong historical + peer signals")

Edge case employees added successfully

Original employees : 2,500
Solid Contributors   : 50
Inconsistent HiPos   : 50
Total employees now  : 2,600

These employees represent:
Solid Contributors → High performer, short tenure,
low OKR, not yet promo eligible
Inconsistent HiPos → Low current performance but
strong historical + peer signals


In [14]:
# ── DIAGNOSTIC CELL — Run this first ──
import pandas as pd
import numpy as np
import os

notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

# Load the FINAL dataset (after all phases)
final_path = os.path.join(
    project_root, 'data', 'processed',
    'employee_final.csv'
)

# Check if file exists
if not os.path.exists(final_path):
    print("employee_final.csv not found")
    print("Try loading employee_features.csv instead")
    final_path = os.path.join(
        project_root, 'data', 'processed',
        'employee_features.csv'
    )

df = pd.read_csv(final_path)

print(f"File loaded: {final_path}")
print(f"Total employees: {len(df):,}")
print()

# ── CHECK 1: Performance Rating distribution ──
print("-" * 55)
print("CHECK 1: Performance Rating Distribution")
print("-" * 55)
print(df['PerformanceRating'].value_counts()
      .sort_index().to_string())
print(f"\nHigh performers (>=4.0): "
      f"{(df['PerformanceRating'] >= 4.0).sum()}")

# ── CHECK 2: PromotionReadinessScore distribution ──
print()
print("-" * 55)
print("CHECK 2: PromotionReadinessScore Distribution")
print("-" * 55)
print(f"Min    : {df['PromotionReadinessScore'].min():.4f}")
print(f"Max    : {df['PromotionReadinessScore'].max():.4f}")
print(f"Mean   : {df['PromotionReadinessScore'].mean():.4f}")
print(f"Median : {df['PromotionReadinessScore'].median():.4f}")
print()
print("Distribution in buckets:")
buckets = pd.cut(
    df['PromotionReadinessScore'],
    bins=[0, 0.10, 0.20, 0.30, 0.35, 0.40,
          0.50, 0.60, 0.65, 0.70, 1.01],
    labels=['0.00-0.10','0.10-0.20','0.20-0.30',
            '0.30-0.35','0.35-0.40','0.40-0.50',
            '0.50-0.60','0.60-0.65','0.65-0.70',
            '0.70-1.00']
)
print(buckets.value_counts().sort_index().to_string())

# ── CHECK 3: The exact combination we need ──
print()
print("-" * 55)
print("CHECK 3: Solid Contributor Candidates")
print("(Rating >= 4.0 AND Readiness < 0.40)")
print("-" * 55)

for threshold in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]:
    count = (
        (df['PerformanceRating'] >= 4.0) &
        (df['PromotionReadinessScore'] < threshold)
    ).sum()
    print(f"   Rating>=4.0 AND Readiness < {threshold:.2f} "
          f"→ {count:>5} employees")

# ── CHECK 4: What IS the readiness range for high performers ──
print()
print("-" * 55)
print("CHECK 4: Readiness Score Range for HIGH Performers only")
print("(Rating >= 4.0)")
print("-" * 55)
high_perf = df[df['PerformanceRating'] >= 4.0]
print(f"   Count : {len(high_perf)}")
print(f"   Readiness Min    : "
      f"{high_perf['PromotionReadinessScore'].min():.4f}")
print(f"   Readiness Max    : "
      f"{high_perf['PromotionReadinessScore'].max():.4f}")
print(f"   Readiness Mean   : "
      f"{high_perf['PromotionReadinessScore'].mean():.4f}")
print(f"   Readiness Median : "
      f"{high_perf['PromotionReadinessScore'].median():.4f}")
print()
print("   Readiness buckets for HIGH PERFORMERS:")
hp_buckets = pd.cut(
    high_perf['PromotionReadinessScore'],
    bins=[0, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 1.01],
    labels=['<0.20','0.20-0.30','0.30-0.40',
            '0.40-0.50','0.50-0.60','0.60-0.70','>0.70']
)
print(hp_buckets.value_counts().sort_index().to_string())

# ── CHECK 5: Show what the DAX thresholds need to be ──
print()
print("-" * 55)
print("CHECK 5: What threshold splits high performers?")
print("-" * 55)
hp_readiness = high_perf['PromotionReadinessScore']
print(f"   To get SOME Solid Contributors we need threshold")
print(f"   above the MINIMUM readiness of high performers")
print(f"   Minimum readiness of any high performer: "
      f"{hp_readiness.min():.4f}")
print(f"   10th percentile: "
      f"{hp_readiness.quantile(0.10):.4f}")
print(f"   25th percentile: "
      f"{hp_readiness.quantile(0.25):.4f}")
print(f"   50th percentile: "
      f"{hp_readiness.quantile(0.50):.4f}")
print()

# ── FINAL ANSWER: What thresholds to use ──
# For Solid Contributor: Rating>=4.0 AND Readiness < threshold
# threshold should be above the minimum readiness of HPs
# Let's use the 30th percentile of HP readiness

p30 = hp_readiness.quantile(0.30)
p60 = hp_readiness.quantile(0.60)

solid_count = (
    (df['PerformanceRating'] >= 4.0) &
    (df['PromotionReadinessScore'] < p30)
).sum()

hp_count = (
    (df['PerformanceRating'] >= 4.0) &
    (df['PromotionReadinessScore'] >= p30) &
    (df['PromotionReadinessScore'] < p60)
).sum()

star_count = (
    (df['PerformanceRating'] >= 4.0) &
    (df['PromotionReadinessScore'] >= p60)
).sum()

print("-" * 55)
print("RECOMMENDED THRESHOLDS FOR YOUR DATA:")
print("-" * 55)
print(f"   Lower threshold (P30) : {p30:.4f}")
print(f"   Upper threshold (P60) : {p60:.4f}")
print()
print(f"   With these thresholds:")
print(f"   Solid Contributors    : {solid_count}")
print(f"   High Performers       : {hp_count}")
print(f"   Stars                 : {star_count}")
print()
print(f"   USE THESE EXACT VALUES IN YOUR DAX FORMULA")
print(f"   Lower: {p30:.3f}")
print(f"   Upper: {p60:.3f}")

File loaded: C:\Users\ganti_kvd0xe3\OneDrive\Sathya\employee_performance_analytics\data\processed\employee_final.csv
Total employees: 2,600

-------------------------------------------------------
CHECK 1: Performance Rating Distribution
-------------------------------------------------------
PerformanceRating
1.0     75
1.5    147
1.6      5
1.7      4
1.8      6
1.9      4
2.0    333
2.1      1
2.2      4
2.3      3
2.4      4
2.5    469
2.6      3
2.7      2
2.8      5
2.9      3
3.0    526
3.5    465
4.0    277
4.1      3
4.2      2
4.3      9
4.4      6
4.5    154
4.6      3
4.7      2
4.8      6
4.9      8
5.0     71

High performers (>=4.0): 541

-------------------------------------------------------
CHECK 2: PromotionReadinessScore Distribution
-------------------------------------------------------
Min    : 0.0290
Max    : 1.0000
Mean   : 0.4678
Median : 0.4570

Distribution in buckets:
PromotionReadinessScore
0.00-0.10     23
0.10-0.20    113
0.20-0.30    289
0.30-0.35    22

In [16]:
# ── GENERATE YOUR PERSONALISED DAX FORMULA ──

# These values come from your diagnostic output above
# The code calculates them automatically

hp_readiness = df[
    df['PerformanceRating'] >= 4.0
]['PromotionReadinessScore']

low_perf_readiness = df[
    df['PerformanceRating'] < 3.0
]['PromotionReadinessScore']

# Thresholds for HIGH performers (Rating >= 4.0)
hp_p30 = round(hp_readiness.quantile(0.30), 3)
hp_p65 = round(hp_readiness.quantile(0.65), 3)

# Thresholds for MID performers (Rating 3.0-4.0)
mid_readiness = df[
    (df['PerformanceRating'] >= 3.0) &
    (df['PerformanceRating'] < 4.0)
]['PromotionReadinessScore']

mid_p30 = round(mid_readiness.quantile(0.30), 3)
mid_p65 = round(mid_readiness.quantile(0.65), 3)

# Thresholds for LOW performers (Rating < 3.0)
low_p40 = round(low_perf_readiness.quantile(0.40), 3)
low_p70 = round(low_perf_readiness.quantile(0.70), 3)

print("-" * 65)
print("YOUR PERSONALISED DAX FORMULA")
print("Copy this EXACTLY into Power BI")
print("-" * 65)

dax_formula = f"""NineBox Label = 
IF(employee_final[PerformanceRating] >= 4.0 && employee_final[PromotionReadinessScore] >= {hp_p65}, "Star",
IF(employee_final[PerformanceRating] >= 4.0 && employee_final[PromotionReadinessScore] >= {hp_p30}, "High Performer",
IF(employee_final[PerformanceRating] >= 4.0, "Solid Contributor",
IF(employee_final[PerformanceRating] >= 3.0 && employee_final[PromotionReadinessScore] >= {mid_p65}, "High Potential",
IF(employee_final[PerformanceRating] >= 3.0 && employee_final[PromotionReadinessScore] >= {mid_p30}, "Core Performer",
IF(employee_final[PerformanceRating] >= 3.0, "Needs Development",
IF(employee_final[PromotionReadinessScore] >= {low_p70}, "Inconsistent HiPo",
IF(employee_final[PromotionReadinessScore] >= {low_p40}, "Improvement Needed",
"Critical Action"))))))))"""

print(dax_formula)
print()
print("-" * 65)
print("VERIFY — Expected counts with these thresholds:")
print("-" * 65)

# Count each category
labels = {
    'Star'              : (df['PerformanceRating'] >= 4.0) & (df['PromotionReadinessScore'] >= hp_p65),
    'High Performer'    : (df['PerformanceRating'] >= 4.0) & (df['PromotionReadinessScore'] >= hp_p30) & (df['PromotionReadinessScore'] < hp_p65),
    'Solid Contributor' : (df['PerformanceRating'] >= 4.0) & (df['PromotionReadinessScore'] < hp_p30),
    'High Potential'    : (df['PerformanceRating'] >= 3.0) & (df['PerformanceRating'] < 4.0) & (df['PromotionReadinessScore'] >= mid_p65),
    'Core Performer'    : (df['PerformanceRating'] >= 3.0) & (df['PerformanceRating'] < 4.0) & (df['PromotionReadinessScore'] >= mid_p30) & (df['PromotionReadinessScore'] < mid_p65),
    'Needs Development' : (df['PerformanceRating'] >= 3.0) & (df['PerformanceRating'] < 4.0) & (df['PromotionReadinessScore'] < mid_p30),
    'Inconsistent HiPo' : (df['PerformanceRating'] < 3.0) & (df['PromotionReadinessScore'] >= low_p70),
    'Improvement Needed': (df['PerformanceRating'] < 3.0) & (df['PromotionReadinessScore'] >= low_p40) & (df['PromotionReadinessScore'] < low_p70),
    'Critical Action'   : (df['PerformanceRating'] < 3.0) & (df['PromotionReadinessScore'] < low_p40),
}

total = 0
for label, condition in labels.items():
    count = condition.sum()
    pct   = count / len(df) * 100
    bar   = "█" * int(pct / 2)
    print(f"{label:<22} {count:>5} ({pct:>5.1f}%) {bar}")
    total += count

print(f"\nTotal: {total:,} (should equal {len(df):,})")

if total == len(df):
    print("All employees assigned to exactly one category")
else:
    print(f"{len(df) - total} employees unassigned")
    print("Check for overlapping conditions") 

-----------------------------------------------------------------
YOUR PERSONALISED DAX FORMULA
Copy this EXACTLY into Power BI
-----------------------------------------------------------------
NineBox Label = 
IF(employee_final[PerformanceRating] >= 4.0 && employee_final[PromotionReadinessScore] >= 0.768, "Star",
IF(employee_final[PerformanceRating] >= 4.0 && employee_final[PromotionReadinessScore] >= 0.591, "High Performer",
IF(employee_final[PerformanceRating] >= 4.0, "Solid Contributor",
IF(employee_final[PerformanceRating] >= 3.0 && employee_final[PromotionReadinessScore] >= 0.515, "High Potential",
IF(employee_final[PerformanceRating] >= 3.0 && employee_final[PromotionReadinessScore] >= 0.448, "Core Performer",
IF(employee_final[PerformanceRating] >= 3.0, "Needs Development",
IF(employee_final[PromotionReadinessScore] >= 0.376, "Inconsistent HiPo",
IF(employee_final[PromotionReadinessScore] >= 0.301, "Improvement Needed",
"Critical Action"))))))))

-------------------------------